In [4]:
import cv2
import json
import random
from pathlib import Path

def calibrate_colors(video_path, num_samples=20, out_path="color_ranges.json"):
    """
    Click the RED dot across random frames, then press 'n'.
    Click the GREEN dot across random frames, then press 'n' or 'q'.
    Saves lower/upper HSV bounds for both colors to `out_path`.
    """
    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        raise RuntimeError(f"Could not read video: {video_path}")

    state = {"color": "red", "samples": {"red": [], "green": []}}

    margin_s = 20
    margin_v = 20
    margin_h = 2

    def get_random_frame():
        frame_number = random.randint(0, total_frames - 1)
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
        ret, frame = cap.read()
        return frame_number, frame

    frame_number, frame = get_random_frame()
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    def on_click(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN:
            h, s, v = hsv[y, x]
            color = state["color"]
            state["samples"][color].append((int(h), int(s), int(v)))

            print(f"[{color}] frame {frame_number}, clicked ({x},{y}) -> HSV: ({h}, {s}, {v})")

            # Pick a new random frame after every click
            show_random_frame()

    def show_random_frame():
        nonlocal frame, hsv, frame_number

        frame_number, frame = get_random_frame()
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        cv2.imshow(window, frame)

    window = "Click RED dot, then press 'n', then click GREEN dot ('q' to finish)"
    cv2.imshow(window, frame)
    cv2.setMouseCallback(window, on_click)

    while True:
        key = cv2.waitKey(1) & 0xFF

        if key == ord("n"):
            if state["color"] == "red":
                state["color"] = "green"
                print("Now click the GREEN dot...")
                show_random_frame()
            else:
                break

        elif key == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

    def build_range(samples, color_name):
        if not samples:
            print(f"WARNING: no samples clicked for {color_name}, skipping")
            return None

        hs = [p[0] for p in samples]
        ss = [p[1] for p in samples]
        vs = [p[2] for p in samples]

        if color_name == "red":
            low_hs = [h for h in hs if h <= 90]
            high_hs = [h for h in hs if h > 90]

            ranges = []

            if low_hs:
                ranges.append([
                    [0, max(0, min(ss) - margin_s), max(0, min(vs) - margin_v)],
                    [min(10, max(low_hs) + margin_h), 255, 255]
                ])

            if high_hs:
                ranges.append([
                    [max(170, min(high_hs) - margin_h), max(0, min(ss) - margin_s), max(0, min(vs) - margin_v)],
                    [180, 255, 255]
                ])

            if not ranges:
                ranges = [
                    [[0, max(0, min(ss) - margin_s), max(0, min(vs) - margin_v)], [10, 255, 255]],
                    [[170, max(0, min(ss) - margin_s), max(0, min(vs) - margin_v)], [180, 255, 255]]
                ]

            return ranges

        else:
            return [[
                [max(0, min(hs) - margin_h), max(0, min(ss) - margin_s), max(0, min(vs) - margin_v)],
                [min(179, max(hs) + margin_h), 255, 255]
            ]]

    result = {}

    for color in ("red", "green"):
        rng = build_range(state["samples"][color], color)
        if rng:
            result[color] = rng

    Path(out_path).write_text(json.dumps(result, indent=2))

    print(f"\nSaved color ranges to {out_path}:")
    print(json.dumps(result, indent=2))

    return result


if __name__ == "__main__":
    calibrate_colors("IMG_6437.MOV", num_samples=20)

[red] frame 468, clicked (764,587) -> HSV: (8, 255, 156)
[red] frame 604, clicked (961,670) -> HSV: (8, 255, 153)
[red] frame 294, clicked (1249,557) -> HSV: (8, 252, 152)
[red] frame 478, clicked (834,635) -> HSV: (8, 255, 150)
[red] frame 413, clicked (973,674) -> HSV: (8, 255, 150)
[red] frame 110, clicked (1226,587) -> HSV: (9, 201, 150)
[red] frame 303, clicked (1058,664) -> HSV: (7, 255, 151)
[red] frame 142, clicked (895,656) -> HSV: (8, 255, 156)
[red] frame 489, clicked (1121,644) -> HSV: (7, 255, 152)
[red] frame 462, clicked (942,667) -> HSV: (8, 255, 152)
[red] frame 437, clicked (1081,661) -> HSV: (8, 255, 151)
[red] frame 612, clicked (946,672) -> HSV: (7, 255, 152)
[red] frame 60, clicked (1346,355) -> HSV: (8, 255, 153)
[red] frame 178, clicked (1286,508) -> HSV: (7, 253, 149)
[red] frame 395, clicked (1258,556) -> HSV: (7, 255, 148)
[red] frame 80, clicked (798,607) -> HSV: (9, 255, 162)
[red] frame 204, clicked (734,568) -> HSV: (7, 207, 159)
[red] frame 165, clicked 